In [1]:
import os
print("Current folder:", os.getcwd())
print("Files in this folder:", os.listdir())

Current folder: /Users/qfl509/Documents/Python/test
Files in this folder: ['main.ipynb', '__pycache__', '.venv', 'utils_TEDS.py', 'test.ipynb']


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline   

# This tells Jupyter: "Every time I run a cell, check if I 
# saved any changes to my helper files and reload them automatically!"

: 

In [ ]:



import numpy as np
import time
import scipy.linalg as la
from types import SimpleNamespace

# Import your custom tools from your new utility module
from utils_TEDS import parList, parSampling, cal_h, cal_jpdf_hist, cal_jFisher, parTran, display_jpdf_design_B3

if __name__ == "__main__":
    
    # --------------------------------------------
    # Options 
    # --------------------------------------------
    Opts = SimpleNamespace()
    Opts.nSampMC = 100
    Opts.Ny = 30             
    Opts.isjpdf = 1          
    Opts.isNorm = 1          
    Opts.isScosine = 0       

    Opts.funName = "design_B3"
    Opts.distType = "Normal"  

    # -------------------------------------------------------------------------
    # (1) Initialization & Sampling
    # -------------------------------------------------------------------------
    RandV = SimpleNamespace()
    RandV.nVar = 10
    RandV.vNominal = np.array([
        [4.8742], [5.4758], [5.6923],
        [4203.00], [4147.68], [5101.48],  
        [2.2], [2.2*0.067], [0.6*0.015],
        [1]
    ])
    RandV.CoV = (1/10) * np.ones((RandV.nVar, 1))

    ListPar, parJ = parList(Opts, RandV, Opts.isNorm)
    nPar = ListPar.shape[0]  
    nS = Opts.nSampMC        

    xS, ListPar, ParSen = parSampling(ListPar, nPar, Opts)  

    # -------------------------------------------------------------------------
    # (2) Evaluate Blackbox function `h`
    # -------------------------------------------------------------------------
    print('Monte Carlo Analysis Starts: ...')
    tic = time.time()
    
    h_Results = cal_h(xS, Opts)
    y = h_Results.y    
    
    elapseTime = round(time.time() - tic, 2)
    print(f'Analysis Completed: {elapseTime}[s]')
     
    # -------------------------------------------------------------------------
    # (3) Post-process for Fisher Information Matrix (FIM)
    # -------------------------------------------------------------------------
    print('Estimating Fisher: ...',flush=True)
    tic = time.time()   
     
    print(' -> Running 2D Histogram...',flush=True)
    yjpdf = cal_jpdf_hist(y[:, [1, 2]], xS, Opts.Ny)

    print(' -> Calculating Raw Fisher Matrix...',flush=True)
    Fraw = cal_jFisher(yjpdf, nPar)
    Fraw = Fraw[0:nPar*2, 0:nPar*2]   

    # --- CRITICAL MAC FIX ---
    # We MUST clear NaNs and Infs BEFORE matrix multiplication in parTran.
    # Otherwise, Apple Silicon Accelerate will crash the kernel.
    if not np.all(np.isfinite(Fraw)):
        print(' -> Cleaning NaNs/Infs from Raw Fisher...')
        Fraw = np.nan_to_num(Fraw, nan=0.0, posinf=0.0, neginf=0.0)

    print(' -> Applying Parameter Transformation...',flush=True)
    Fn, b_v = parTran(Fraw, ListPar, parJ, Opts.isNorm)

    # (Optional) Save to a MATLAB file right before the risky Eigen analysis!
    import scipy.io as sio
    sio.savemat('debug_workspace.mat', {'Fn': Fn, 'Fraw': Fraw})
    print(' -> (Workspace variables temporarily saved to debug_workspace.mat)')

    print(' -> Running Eigen Analysis...',flush=True)
    lambda_val, V_e = la.eigh(Fn)

    EigIndex = np.argsort(lambda_val)[::-1]
    V_e = V_e[:, EigIndex]
    lambda_val = lambda_val[EigIndex]
    D_e = np.diag(lambda_val)    
     
    elapseTime = round(time.time() - tic, 2)
    print(f'Fisher EigenAnalysis Completed: {elapseTime}[s]')
     
    # -------------------------------------------------------------------------
    # (4) Display Results
    # -------------------------------------------------------------------------
    Opts.displayMode = 1
    disp_h_name = f"display_jpdf_{Opts.funName}"
    
    try:
        # Attempts to find the specific display function if it exists
        disp_h = globals()[disp_h_name]
        disp_h(y, yjpdf, D_e, V_e, nPar, Opts)
    except KeyError:
        print(f"Warning: Display function '{disp_h_name}' not defined in workspace.")

Monte Carlo Analysis Starts: ...


Evaluating Blackbox Function: 100%|██████████| 100/100 [00:00<00:00, 13524.34it/s]

Analysis Completed: 0.02[s]
Estimating Fisher: ...
 -> Running 2D Histogram...
 -> Calculating Raw Fisher Matrix...
